# Project interactive visualization
Using a dataset that your group is consider using for the term project, let's build some meaningful user-driven data visualization. Depending on your dataset this could include:
- Usage of geospatial information
- Building interactive views with widgets
- Organize multiple components into a simple dashboard

Keep these design principles in mind:
While you navigate through this notebook some things to take into consideration:
* Do not add interaction just to add it. Make sure it helps answer a question.
* Use meaningful titles and labels
* Document your code so it's readable and clean. If something does not work, document the issue and explain your best attempt.

In [ ]:
!pip install jupyter_bokeh

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from geopy.geocoders import Nominatim
import panel as pn
pn.extension('plotly')

In [ ]:
### Load your dataset here

# Example:
# df = pd.read_csv("your_dataset.csv")
#data import
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Write your code here
team_stats = pd.read_csv('/content/drive/MyDrive/NBA Stats/nba_team_stats_advanced.csv')
player_stats = pd.read_csv('/content/drive/MyDrive/NBA Stats/nba_player_stats.csv')
quarter_scores = pd.read_csv('/content/drive/MyDrive/NBA Stats/nba_quarter_scores.csv')

In [ ]:
team_stats.columns

Index(['gameId', 'teamCity', 'teamName', 'gameType', 'season', 'astPct',
       'astRatio', 'astTo', 'defRating', 'drebPct', 'eDefRating', 'eNetRating',
       'eOffRating', 'ePace', 'efgPct', 'gameDate', 'home', 'matchup',
       'netRating', 'offRating', 'opponentTeamCity', 'opponentTeamId',
       'opponentTeamName', 'orebPct', 'pace', 'pacePer40', 'pie', 'poss',
       'rebPct', 'teamAbbreviation', 'tmTovPct', 'tsPct', 'wl'],
      dtype='object')

In [ ]:
player_stats.columns

Index(['gameId', 'gameDate', 'season', 'gameType', 'matchup', 'home', 'wl',
       'personId', 'firstName', 'lastName', 'teamCity', 'teamName',
       'teamAbbreviation', 'position', 'minutes', 'fgm', 'fga', 'fgPct',
       'fg3m', 'fg3a', 'fg3Pct', 'ftm', 'fta', 'ftPct', 'oreb', 'dreb', 'reb',
       'ast', 'stl', 'blk', 'tov', 'pf', 'pts', 'plusMinus', 'age', 'height',
       'heightInches', 'weight', 'college', 'country', 'draftYear',
       'draftRound', 'draftNumber'],
      dtype='object')

In [ ]:
quarter_scores.columns

Index(['gameId', 'gameDate', 'season', 'gameType', 'matchup', 'home', 'wl',
       'teamCity', 'teamName', 'teamAbbreviation', 'ptsQ1', 'ptsQ2', 'ptsQ3',
       'ptsQ4', 'ptsFinal'],
      dtype='object')

## Question 1: Analytical Question
Write a question about your data that can be explored with an interactive plot. A good example would be a question where you can compare different categories. Explain how the plot help answer your question and why you choose that plot type.

Examples:
- Which regions have the highest average values?
- How does one variable compare across time?

Question: How does a team’s offensive rating relate to its assist-to-turnover ratio, and how do these differ between wins and losses?

In [ ]:
f = px.scatter(
    team_stats,
    x = 'offRating',
    y = 'astTo',
    color = 'wl',
    hover_data = ['teamName', 'gameDate'],
    title = 'Offensive Rating vs Assist to Turnover Ratio in Wins and Losses',
    labels = {'offRating': 'Offensive Rating', 'astTo': 'Assist to Turnover Ratio'}
)
f.show()

This scatter plot helps visualize the relationship between offensive rating and assist-to-turnover ratio while distinguishing between wins and losses using color. It shows whether higher efficiency in ball movement (assist-to-turnover ratio) is associated with stronger offensive performance and better outcomes. A scatter plot is appropriate because it allows us to observe correlations between two continuous variables while also comparing categorical outcomes (win vs. loss).

## Question 2. Create a simple interaction plot
Create a plot, it can be related to your question #1 or a new question, that users can interact with in some meaningful way. Explain in a markdown, how does the interaction add to the analysis.

Example of possible interactions:
*   Hover over information
*   Toogle between groups

In [ ]:
import plotly.graph_objects as go

In [ ]:
cleaner = team_stats.dropna(subset=['offRating', 'astTo', 'wl', 'gameType'])

In [ ]:
all = px.scatter(
    cleaner,
    x = 'offRating',
    y = 'astTo',
    color = 'wl',
    hover_data = ['teamName', 'gameDate']
)

In [ ]:
regular = px.scatter(
    cleaner[cleaner['gameType'] == 'Regular Season'],
    x = 'offRating',
    y = 'astTo',
    color = 'wl',
    hover_data = ['teamName', 'gameDate']
)

In [ ]:
playoff = px.scatter(
    cleaner[cleaner['gameType'] == 'Playoffs'],
    x = 'offRating',
    y = 'astTo',
    color = 'wl',
    hover_data = ['teamName', 'gameDate']
)

In [ ]:
fig = go.Figure()

for trace in all.data:
    trace.visible = True
    fig.add_trace(trace)
for trace in regular.data:
    trace.visible = False
    fig.add_trace(trace)
for trace in playoff.data:
    trace.visible = False
    fig.add_trace(trace)

n = len(all.data)

In [ ]:
fig.update_layout(
    title = 'Offensive Rating vs Assist to Turnover Ratio in Wins and Losses (All Games)',
    xaxis_title = 'Offensive Rating',
    yaxis_title = 'Assist to Turnover Ratio',
    updatemenus = [
        dict(
            buttons = [
                dict(
                    label = 'All Games',
                    method = 'update',
                    args = [
                        {'visible': [True]*n + [False]*n + [False]*n},
                        {'title': 'Offensive Rating vs Assist to Turnover Ratio in Wins and Losses (All Games)'}
                    ]
                ),
                dict(
                    label = 'Regular Season',
                    method = 'update',
                    args = [
                        {'visible': [False]*n + [True]*n + [False]*n},
                        {'title': 'Offensive Rating vs Assist to Turnover Ratio in Wins and Losses (Regular Season)'}
                    ]
                ),
                dict(
                    label = 'Playoffs',
                    method = 'update',
                    args = [
                        {'visible': [False]*n + [False]*n + [True]*n},
                        {'title': 'Offensive Rating vs Assist to Turnover Ratio in Wins and Losses (Playoffs)'}
                    ]
                )
            ],
            direction = 'down',
            showactive = True,
            x = 0.1,
            y = 1.1
        )
    ]
)

fig.show()

The interaction lets users switch between all games, regular season, and playoffs, making it easier to compare how team performance changes in each context. Instead of looking at everything at once, users can focus on specific subsets and see if the relationship between offensive rating and assist-to-turnover ratio differs across game types. This makes the analysis clearer and more useful.


## Question 3. Choropleth Planning
Design a choropleth idea using your dataset.
In a markdown cell:
*  Identify the geographic unit you would map (state, county, country, ZIP code, etc.)
*  Identify the variable you would color by
*  Explain if any aggregation or preprocessing is needed
*  Briefly describe what GeoJSON file would be required

You do not need to have a perfect dataset for this question. However, your plan should be realistic.

If your data does not fully support a choropleth, build a prototype table that explains that structure you would need before mapping.

Our data doesn't fully support choropleth, but this is how we would plan it out:

- Geogrphic Unit: NBA team location -> each city would be mapped out
- Variable to color by: Average Net Rating per team
- Aggregation/Reprocessing: Data would need to be grouped by 'teamName', and each grouping would have to contain the average Net Rating across the team's games.
- GeoJSON: U.S. States GeoJSON file
- Note: Point-based map may actually be better because there can be multiple teams in one state, but this would still show geographic tendencies.

## Question 4. Geospatial Possibility Check

Determine whether your dataset can support a map-based visualization.

In a markdown cell, answer one of the following:
- If **yes**, explain what geographic fields you have and what type of map is appropriate.
- If **no**, explain what is missing and what you would need to create a map.

Write code that inspects or prepares the geographic column(s) you may use.

The dataset can support a basic map since it includes team city names, which can be converted into latitude and longitude for a point-based visualization. However, a map may not be very useful here because team performance metrics like offensive rating or net rating are not strongly related to location. Also, since multiple teams can be in the same state, a region-based map could hide important differences. So while mapping is possible, it is not the most meaningful way to analyze this data.

In [ ]:
#should be 29
print("Unique team cities:")
print(team_stats['teamCity'].unique())
print(f"Total unique cities: {team_stats['teamCity'].nunique()}")

team_stats[['teamCity', 'teamName', 'opponentTeamCity', 'opponentTeamName']].drop_duplicates().head(10)



Unique team cities:
['Boston' 'New York' 'Los Angeles' 'Minnesota' 'Indiana' 'Detroit'
 'Atlanta' 'Brooklyn' 'Miami' 'Orlando' 'Philadelphia' 'Milwaukee'
 'Cleveland' 'Toronto' 'Charlotte' 'Houston' 'Chicago' 'New Orleans'
 'Memphis' 'Utah' 'Phoenix' 'Portland' 'San Francisco' 'Washington'
 'Dallas' 'San Antonio' 'Denver' 'Oklahoma City' 'Sacramento']
Total unique cities: 29


,teamCity,teamName,opponentTeamCity,opponentTeamName
0,Boston,Celtics,New York,Knicks
1,New York,Knicks,Boston,Celtics
2,Los Angeles,Lakers,Minnesota,Timberwolves
3,Minnesota,Timberwolves,Los Angeles,Lakers
4,Indiana,Pacers,Detroit,Pistons
5,Detroit,Pistons,Indiana,Pacers
6,Atlanta,Hawks,Brooklyn,Nets
7,Brooklyn,Nets,Atlanta,Hawks
8,Miami,Heat,Orlando,Magic
9,Orlando,Magic,Miami,Heat


## Question 5. Geopy / Location Preparation

If your dataset has location names, addresses, cities, states, or countries, use geopy on a sample of your data to geocode locations or validate location information.

If your dataset does not have data that contains location, pick 5 destination you want to visit and use geopy to geocode locations.

In [ ]:
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="nba_project")

sample_cities = team_stats['teamCity'].unique()[:5]

geo_results = []
for city in sample_cities:
    location = geolocator.geocode(city + ", USA")
    if location:
        geo_results.append({
            'city': city,
            'latitude': location.latitude,
            'longitude': location.longitude
        })
    time.sleep(1)

geo_df = pd.DataFrame(geo_results)
geo_df

,city,latitude,longitude
0,Boston,42.358834,-71.057830
1,New York,40.712728,-74.006015
2,Los Angeles,34.053691,-118.242766
3,Minnesota,45.989659,-94.611329
4,Indiana,40.327013,-86.174693


## Question 6. Panel Widget

Create a Panel Widget that controls something in your analysis such as the ability to choose a column, category, year, etc.

The widget should affect an output such as a plot, table, or summary statistic.

In [ ]:
teams = sorted(team_stats['teamName'].unique())
team_widget = pn.widgets.Select(name='Select Team', options=list(teams))

def make_plot(team):
    filtered = team_stats[team_stats['teamName'] == team]
    fig = px.scatter(
        filtered,
        x='pace',
        y='offRating',
        color='wl',
        hover_data=['gameDate', 'matchup'],
        title=f'{team} - Pace vs Offensive Rating',
        labels={'pace': 'Pace', 'offRating': 'Offensive Rating'}
    )
    return fig

interactive = pn.bind(make_plot, team=team_widget)

pn.Column(team_widget, pn.pane.Plotly(interactive))

Column
    [0] Select(options=['76ers', 'Bucks', ...], value='76ers')
    [1] Plotly(Figure)

## Question 7. Mini Dashboard

Build a small Panel dashboard with at least two components. Arrange the components so that it is in a readable layout. Your dashboard should include:
* At least one plot,
* An additional element of your choice such as a widget, table, second plot, etc.

In [ ]:
teams = sorted(team_stats['teamName'].unique())
team_widget = pn.widgets.Select(name='Select Team', options=list(teams))

def make_plot(team):
    filtered = team_stats[team_stats['teamName'] == team]
    fig = px.scatter(
        filtered,
        x='pace',
        y='offRating',
        color='wl',
        hover_data=['gameDate', 'matchup'],
        title=f'{team} - Pace vs Offensive Rating',
        labels={'pace': 'Pace', 'offRating': 'Offensive Rating'}
    )
    return fig

def make_table(team):
    filtered = team_stats[team_stats['teamName'] == team].sort_values('gameDate', ascending=False)
    display_cols = ['gameDate', 'matchup', 'wl', 'offRating', 'defRating', 'pace']
    df = filtered[display_cols].head(10)
    styled_df = df.style.set_properties(**{'background-color': 'white', 'color': 'black', 'border': '1px solid black'}) \
        .set_table_styles([{'selector': 'th', 'props': [('background-color', 'lightgrey'), ('color', 'black'), ('border', '1px solid black')]}])

    return pn.pane.DataFrame(styled_df, width=500)

interactive_plot = pn.bind(make_plot, team=team_widget)
interactive_table = pn.bind(make_table, team=team_widget)
dashboard = pn.Column(
    "# Team Performance Dashboard",
    team_widget,
    pn.Row(
        pn.pane.Plotly(interactive_plot, width=600),
        pn.Column(
            "### Recent 10 Games",
            interactive_table
        )
    )
)

dashboard

Column
    [0] Markdown(str)
    [1] Select(options=['76ers', 'Bucks', ...], value='76ers')
    [2] Row
        [0] Plotly(Figure, width=600)
        [1] Column
            [0] Markdown(str)
            [1] ParamFunction(function, _pane=DataFrame, defer_load=False)

## Question 8. Reflection

Write a short reflection addressing all of the following:
- Which interactive element was most useful in your notebook?
- What was the hardest part of working with your dataset?
- Did implementing interactive tools help your analysis? Why or why not?
- If you had more time, what would you improve or add?

1. **Which interactive element was most useful in your notebook?**

    I would say the most useful interactive element in our notebook was definetly the interactive plotly mini-dashboard with the dropdown menu, as it was the most useful element. It allowed us to seamlessly filter the scatter plot and recent game data just by selecting our favorite teams without needing to rerun the whole cells, which makes analysis much faster especially on the long run.

2. **What was the hardest part of working with your dataset?**

    The hardest part wasn't necessarily the data itself, but rather managing the interactive widget enviroments within Google Colab. For example, getting python backed widgets (like panel) to syncronizee with the frontend was extremly frustrating due to connectivity issues. This made me extremely mad but in the end we got it to work.

3. **Did implementing interactive tools help your analysis? Why or why not?**

    Implementing interactive tools absolutely helped in our analysis. NBA data is extremely dense with 30 teams nad each has over 80 games and on top of that there are other statistics that needs to be covered. Static plots with all teams combined would become cluttered and honeslty it would be a big unreadable mess. Interactive tools like hover data, group toggles, and team dropdowns allowed us to isolate specific variables and easily spot trends like highest offensive ratings, or how specific pace impacts offensive rating.

4. **If you had more time, what would you improve or add?**

    If we had more time (it was spring break so we started working on this over the weekend), we would fully implement geographic visualization to see if there are regional trends in team performance like net rating for example. Additionally, we would integrate the player_stats and quarter_scores dataset into dashboard, which would allow us to click on a specific game in the team view and drill down into individual player performances or quarter-by-quarter breakdowns.